In [25]:

import numpy as np
import pandas as pd
import pickle as p
import tensorflow as tf



In [26]:
# 1. Load trained model
model = tf.keras.models.load_model("model.h5")

In [27]:
# 2. Load encoders and scaler
with open("label_gender.pkl", "rb") as file:
    label_gender = p.load(file)


In [28]:
with open("one_hot_geo.pkl", "rb") as file:
    one_hot_geo = p.load(file)


In [29]:

with open("scaler.pkl", "rb") as file:
    scaler = p.load(file)

In [30]:
# 3. Sample customer data
input_data = {
    "CreditScore": 600,
    "Geography": "France",
    "Gender": "Male",
    "Age": 40,
    "Tenure": 3,
    "Balance": 60000,
    "NumOfProducts": 2,
    "HasCrCard": 1,
    "IsActiveMember": 1,
    "EstimatedSalary": 50000
}

In [51]:
geo_encoded=one_hot_geo.transform([[input_data['Geography']]])

d:\DEMO\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [52]:
geo_encoded.toarray()

array([[1., 0., 0.]])

In [31]:
# 4. Convert input to DataFrame
input_df = pd.DataFrame([input_data])

In [32]:
# 5. Encode Gender
input_df["Gender"] = label_gender.transform(
    input_df["Gender"]
)

In [53]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [33]:
# 6. Encode Geography
geo_encoded = one_hot_geo.transform(
    input_df[["Geography"]]
)

In [54]:
geo_encoded

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1 stored elements and shape (1, 3)>

In [34]:
if hasattr(geo_encoded, "toarray"):
    geo_encoded = geo_encoded.toarray()


In [35]:
geo_columns = one_hot_geo.get_feature_names_out(
    ["Geography"]
)

In [36]:
if hasattr(geo_encoded, "toarray"):
    geo_encoded = geo_encoded.toarray()


In [37]:
geo_columns = one_hot_geo.get_feature_names_out(
    ["Geography"]
)


In [56]:
geo_columns

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [38]:
geo_df = pd.DataFrame(
    geo_encoded,
    columns=geo_columns
)


In [57]:
geo_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [39]:
# 7. Remove original Geography and combine encoded columns
input_df.drop(columns=["Geography"], inplace=True)

input_df = pd.concat(
    [input_df.reset_index(drop=True), geo_df],
    axis=1
)

In [40]:
# 8. Match scaler's training feature order
if hasattr(scaler, "feature_names_in_"):
    expected_columns = list(scaler.feature_names_in_)


In [41]:

missing = [
        col for col in expected_columns
        if col not in input_df.columns
    ]


In [42]:
extra = [
        col for col in input_df.columns
        if col not in expected_columns
    ]

In [43]:

if missing or extra:
        raise ValueError(
            f"Feature mismatch. Missing: {missing}; Extra: {extra}"
        )


In [44]:
input_df = input_df[expected_columns]

In [59]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [45]:
# 9. Scale input
input_scaled = scaler.transform(input_df)


In [46]:
# 10. Predict churn probability
pred = model.predict(input_scaled, verbose=0)
prediction_probability = float(np.asarray(pred).reshape(-1)[0])

print(f"Churn Probability: {prediction_probability:.2%}")


Churn Probability: 3.68%


In [60]:
pred

array([[0.03678859]], dtype=float32)

In [47]:
# 11. Display prediction
if prediction_probability >= 0.5:
    print("Customer is likely to leave.")
else:
    print("Customer is likely to stay.")


Customer is likely to stay.
